# Task B — TME-inspired Residual Adapter Trial
Thử adapter VisFit trên Conventional-only seed42 đã huấn luyện và **đóng băng**. Không phải TME gốc FashionERN. Không official-val, không final benchmark, không train encoder/head XAA/DVR. Chỉ sàng lọc 2.048 development-train / đủ 1.796 development-val, gallery 598/599/599; seed42, tối đa3epoch.

Nguồn cảm hứng: [CLIP-Adapter](https://arxiv.org/html/2110.04544v2), bottleneck và residual sau encoder đóng băng. RI/TA, cross-attention, gate và cosine objective dưới đây là thiết kế thử nghiệm VisFit, không phải sao chép loss/kiến trúc bài báo.

## Review nguồn đã kiểm tra
Đã đọc code/output notebook Pilot, Development, Preparation, XAA/DVR; contract/config/mapping và artifact thật trên Drive. Conventional best epoch3, 12.153step, Mean **72,6305734380787%** là **mốc artifact cũ để đối chiếu**, không phải output notebook mới. Checkpoint SHA `f62513032c086ce5c2e09a9774add5bc590e6ded81c5b9d92d0229cd526f080b`; schema lưu `signature/model/optimizer/rng/config/cache_fingerprint`. Class kiến trúc cũ chỉ dùng để strict-load đầy đủ state, sau đó bỏ RI/TA/merge cũ, giữ text/projection conventional đóng băng.

Bootstrap theo dependency owner; không nâng torch/CUDA. Nếu module bản khác đã nạp: lưu log/notebook rồi restart **một lần**, chạy lại từ đầu. Cache không dùng text Fashion-CLIP cũ, không đọc cache 12 lớp, không JSON/hash Drive trong batch. Một caption rỗng vẫn warning, giữ `.strip('.?, ')`, thứ tự và query ID đã khóa. Không chỉnh annotation hoặc tạo caption.


In [ ]:
# Cấu hình phải giữ nguyên khi resume; đổi batch/loss/precision cần experiment_id mới.
CFG = {'preparation_root': '/content/drive/MyDrive/ĐATN/data/processed/benchmark_preparation/FashionIQ/fashioniq_prep_v1_20261103', 'development_root': '/content/drive/MyDrive/ĐATN/data/processed/tme_development/FashionIQ/tme_dev_v1_seed42', 'output_root': '/content/drive/MyDrive/ĐATN/data/processed/tme_adapter_trial/FashionIQ', 'experiment_id': 'adapter_trial_v1_seed42', 'seed': 42, 'subset_size': 2048, 'max_epochs': 3, 'batch_size': 32, 'profile_batches': [32, 64], 'encode_batch_size': 16, 'eval_batch_size': 32, 'cache_shard_size': 128, 'adapter_width': 128, 'adapter_heads': 4, 'learning_rate': 0.0001, 'weight_decay': 0.01, 'retrieval_temperature': 0.07, 'alignment_weight': 0.1, 'gradient_clip': 1.0, 'checkpoint_every_steps': 16, 'resume': True, 'amp': False, 'tf32': False, 'require_cuda': True, 'baseline_max_metric_deviation_pp': 0.01, 'gate_equivalence_atol': 1e-06, 'local_root': '/content/visfit_tme_adapter_trial', 'vision_dim': 768}

## 1. Bootstrap, Drive và định nghĩa
Dependencies chuẩn bị trước import model. Khi Drive yêu cầu quyền, hoàn tất trong Colab; không ghi credential vào notebook/output.

In [ ]:
# Chỉ stdlib trước import model; bootstrap fix: dependency-owner-and-loaded-version-v2.
import sys, subprocess, importlib.util, importlib.metadata, json
from pathlib import Path
BOOTSTRAP = {'python':sys.version,'installs':[],'pins':{},'passed':False,'bootstrap_fix':'dependency-owner-and-loaded-version-v2'}
local=Path('/content/visfit_tme_adapter_bootstrap');local.mkdir(parents=True,exist_ok=True)
BOOTSTRAP['loaded_before_bootstrap']={k:{'version':getattr(v,'__version__',None),'path':getattr(v,'__file__',None)} for k,v in list(sys.modules.items()) if k in ('torch','transformers','tokenizers','huggingface_hub')}
PINS = {'transformers':'4.57.1','tokenizers':'0.22.1','huggingface-hub':'0.36.0'}
for package in ('torch','numpy','pandas','Pillow'):
    try: BOOTSTRAP['pins'][package]=importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        raise RuntimeError('Colab base package missing: '+package+'; use a standard GPU runtime') from None
needed=[]
for package,version in PINS.items():
    try: installed=importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError: installed=None
    if installed!=version:needed.append(package+'=='+version)
    module={'huggingface-hub':'huggingface_hub'}.get(package,package)
    loaded=any(n==module or n.startswith(module+'.') for n in sys.modules)
    loaded_version=getattr(sys.modules.get(module),'__version__',None)
    if loaded and (installed!=version or loaded_version!=version):
        BOOTSTRAP.update(restart_required=True,blocked_package=package,installed_version=installed,loaded_version=loaded_version)
        (local/'bootstrap_report.json').write_text(json.dumps(BOOTSTRAP,indent=2))
        raise RuntimeError('A dependency requiring a version change is already loaded: '+package+'. Save the notebook and use Runtime > Restart session, then run from the top. No automatic install/restart loop.')
    BOOTSTRAP['pins'][package]=version
if needed:
    p=subprocess.run([sys.executable,'-m','pip','install','--quiet',*needed],capture_output=True,text=True)
    if p.returncode:raise RuntimeError('Dependency installation failed; stopped before importing models. No torch/CUDA upgrade requested.')
    BOOTSTRAP['installs']=needed
check=subprocess.run([sys.executable,'-m','pip','check'],capture_output=True,text=True)
BOOTSTRAP['pip_check']={'returncode':check.returncode,'report':check.stdout+check.stderr}
# Scope by dependent package owner, not a substring of the dependency mentioned in the error.
# Gradio/Diffusers are not used by this pilot; their conflicts remain visible in the report.
from packaging.requirements import Requirement
from packaging.utils import canonicalize_name
pending=['torch','torchvision','transformers','tokenizers','huggingface-hub','numpy','pandas','Pillow','safetensors','packaging']
core=set()
while pending:
    owner=canonicalize_name(pending.pop())
    if owner in core:continue
    core.add(owner)
    for raw in importlib.metadata.requires(owner) or []:
        req=Requirement(raw)
        if req.marker is None or req.marker.evaluate({'extra':''}):pending.append(req.name)
lines=[line for line in (check.stdout+check.stderr).splitlines() if line.strip() and 'No broken requirements found' not in line]
critical=[line for line in lines if canonicalize_name(line.split()[0]) in core]
BOOTSTRAP['dependency_scope']={'core_packages':sorted(core),'critical_conflicts':critical,'outside_pilot_conflicts':[line for line in lines if line not in critical]}
local=Path('/content/visfit_tme_adapter_bootstrap');local.mkdir(parents=True,exist_ok=True)
(local/'bootstrap_report.json').write_text(json.dumps(BOOTSTRAP,indent=2))
if check.returncode and critical:raise RuntimeError('pip check found TME dependency conflicts: '+'; '.join(critical))
probe=subprocess.run([sys.executable,'-c',
    'import torch,transformers; from transformers import CLIPModel,CLIPTokenizer,CLIPImageProcessor; from transformers.generation import GenerationMixin; print(transformers.__version__); print(transformers.__file__)'],capture_output=True,text=True)
BOOTSTRAP['import_probe']={'returncode':probe.returncode,'stdout':probe.stdout,'traceback':probe.stderr}
(local/'bootstrap_report.json').write_text(json.dumps(BOOTSTRAP,indent=2))
if probe.returncode:raise RuntimeError('CLIP dependency import failed in a clean subprocess. Full traceback: /content/visfit_tme_adapter_bootstrap/bootstrap_report.json. Diagnose before dataset/training.')
BOOTSTRAP['passed']=True
(local/'bootstrap_report.json').write_text(json.dumps(BOOTSTRAP,indent=2))
print('Bootstrap đạt; giữ nguyên torch/CUDA:',BOOTSTRAP['pins']['torch'])
print('TME dependency conflicts:',len(critical),'| clean CLIP import:',probe.returncode==0,'| packages installed:',BOOTSTRAP['installs'])
if check.returncode:print('pip check có xung đột ngoài dependency graph của pilot; giữ nguyên đầy đủ trong bootstrap_report.json. Không coi toàn môi trường là pip-clean.')


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
PREPARATION_ANCHORS = {'data_audit.json': 'fbaa7eae7b99c5e5aeeddcb0bc6027d3870e8c4f9be5658559caa95662321e95', 'feature_metadata.json': '48ee1866dc22f07fa588ea815318ddc9995cfa8734ffd0f2acc88c430f855342', 'extraction_contract.json': 'bbd53b94172aa5c07a358a0697163c620a823b297c6fec6209167bdb59d1f0a1', 'training_plan.json': 'e5496feae33ab0e9dc832e29a1a9097b2d7fed6c2683bd9d4314d6e4973bdc21', 'development_gallery.csv': '83c08f14b606227c1a406b5074163ef70e42eb025144d19e2b1de4f38d234dc5', 'development_val.csv': 'ace52e94935adacfd19aeaea7a99314602954bce9bff43458ceb564fdf05a8b5', 'protocol_config.json': '2fa2322dee2ebe9964e4ce63ebf0cd8dabad83392a39e288d0e43334e40faffb', 'development_train.csv': '465f758604249e2005cc58c0dfa79edf236081613e908c1e5942fd61abf17086', 'feature_views/development_gallery.csv': '283199425c5fc88fb8e063ea28be629e611e44513689e97f22ee3d0ec5a33927', 'feature_views/development_val.csv': 'cbd8d7b72fa7fb4e3c38d64f3bb8ceeeba73be946e9918c5a77c856a42217619', 'feature_views/development_train.csv': '6f7a5e6ab8792dd8fd08d6fea8c472a9d9b56b59af38ba146467d797f8a98ff0'}

SOURCE_LOCKS = {'config.json': 'b3d48c8550a8f36ee1bbbc575878d10ac10e416509b8a599ba23019c7561935f', 'best.json': '9ec6117b48e0df64e49247d9f41da1cc035c1a504e00551abc1aad654aadbca1', 'architecture_contract.json': '49c7db63abcb71ce0d7be083aa3a7b82c89e290adabbd77264226e9ee31b627b', 'pilot_config.json': '3b6097620b76a2789579375987e64594fc82637e7e2a3ab271f9ab4145243685', 'runtime_contract.json': '02696d2a36995ffd90791671203673b689bede949c5f05d92c20c4741e65ee2d', 'cache_binding.json': '78f3e73eed98fa2c6b27edd61fe16c542f925a9157cced035e8dd6aef858093d', 'pilot_manifest_hashes.json': '6d8198b224e4e07242bbc39dcef7e189e06b9a1d50a416f6e5013572debe657c', 'metric_history.json': '253717bafb9c38d8dd1f0cf484d78b6165c2708233afec225bc7a905e3d8829e', 'epoch_03_per_query.csv': '00712dcc75ac63ded64e75465e566c1c183365eda3df7b7f832ddc0f411568a2'}
EXPECTED_BASE_MARKER = {'phase': 'conventional', 'sha256': 'f62513032c086ce5c2e09a9774add5bc590e6ded81c5b9d92d0229cd526f080b', 'signature': 'c6cf0642cdcf5b7c7c5fc8fe6246fd7f0c668377036fefb12d45cfb1c10dc3d3', 'step': 12153}
EXPECTED_METRICS = {'Mean': 72.6305734380787, 'gallery_counts': {'dress': 598, 'shirt': 599, 'toptee': 599}, 'n_query': 1796, 'overall': {'R@1': 25.833096037059164, 'R@10': 61.020876488685154, 'R@5': 49.44016690768524, 'R@50': 84.24027038747225}, 'per_category': {'dress': {'R@1': 22.073578595317723, 'R@10': 54.515050167224075, 'R@5': 43.97993311036789, 'R@50': 79.76588628762542}, 'shirt': {'R@1': 27.545909849749584, 'R@10': 61.769616026711184, 'R@5': 51.41903171953256, 'R@50': 84.47412353923205}, 'toptee': {'R@1': 27.879799666110184, 'R@10': 66.7779632721202, 'R@5': 52.92153589315526, 'R@50': 88.48080133555926}}, 'query_counts': {'dress': 598, 'shirt': 599, 'toptee': 599}, 'scope': 'full locked development validation; NOT official-val', 'search': 'exact L2 cosine FP32, reference retained, exact ties original gallery_index ascending', 'unit': 'percent'}

CODE_SHA256 = '17ae2f4fe46c69804c8fa405751a069ffc4cbe99a1ff9498cdff01aea1382a3a'

In [ ]:
# common.py — code nhúng, không phụ thuộc file local
import os, sys, json, hashlib, math, random, time, copy, traceback, uuid, shutil
import platform, tempfile, contextlib, importlib.metadata, subprocess
from pathlib import Path
from collections import OrderedDict
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.nn import functional as F
from PIL import Image
from transformers import CLIPModel, CLIPTokenizer, CLIPImageProcessor

CATEGORIES = ('dress', 'shirt', 'toptee')
CAPTION_POLICY = 'clip4cir-strip-dot-question-comma-space-capitalize-first-and-v1'
IMPLEMENTATION = 'visfit-tme-adapter-trial-v1.0.0'

def require(condition, message):
    if not condition: raise RuntimeError(message)

def canonical(x):
    return json.dumps(x, ensure_ascii=False, sort_keys=True, separators=(',', ':'), allow_nan=False).encode()

def fingerprint(x): return hashlib.sha256(canonical(x)).hexdigest()

def file_hash(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda: f.read(8*1024*1024), b''): h.update(block)
    return h.hexdigest()

def read_json(path): return json.loads(Path(path).read_text())

def write_bytes(path, raw, immutable=False):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    if immutable and path.exists():
        require(path.read_bytes() == raw, 'Existing immutable artifact differs: '+str(path)); return
    tmp = path.with_name(path.name+'.'+uuid.uuid4().hex+'.partial')
    tmp.write_bytes(raw); os.replace(tmp, path)

def write_json(path, value, immutable=False):
    write_bytes(path, canonical(value)+b'\n', immutable)

def write_csv(path, frame, immutable=False):
    write_bytes(path, frame.to_csv(index=False, lineterminator='\n').encode(), immutable)

def safe_child(root, relative):
    root = Path(root).resolve(); path = (root/relative).resolve()
    require(path != root and root in path.parents, 'Path escapes root'); return path

def tensor_hash(module):
    h = hashlib.sha256()
    for name, value in sorted(module.state_dict().items()):
        h.update(name.encode()); h.update(str(value.dtype).encode())
        h.update(value.detach().cpu().contiguous().numpy().tobytes())
    return h.hexdigest()

def finite(x, name):
    require(bool(torch.isfinite(x).all()), 'NaN/Inf: '+name); return x

def unit(x):
    finite(x, 'unnormalized vector')
    require(bool((x.float().norm(dim=-1)>1e-8).all()), 'Zero retrieval vector')
    return F.normalize(x.float(), dim=-1)

def normalize_caption(x): return x.strip('.?, ')

def caption_text(a, b): return normalize_caption(a).capitalize()+' and '+normalize_caption(b)

def assess_captions(caps):
    if not isinstance(caps, list) or len(caps)!=2 or not all(isinstance(x,str) for x in caps):
        return 'schema_error', None
    content = [bool(normalize_caption(x)) and not normalize_caption(x).isspace() for x in caps]
    if not any(content): return 'both_empty_captions', None
    return None, 'single_empty_caption' if not all(content) else None

def stratified_sample(frame, n, seed):
    require(type(n) is int and 3<=n<=len(frame), 'Sample budget outside available range; change CFG explicitly')
    require(frame.query_id.is_unique, 'Duplicate query IDs')
    # Sort by query ID before RNG: stable if the input CSV order changes.
    rng = np.random.default_rng(seed); chosen=[]
    quotas = {c:n//3+int(i<n%3) for i,c in enumerate(CATEGORIES)}
    for c in CATEGORIES:
        ids=sorted(frame.loc[frame.category==c,'query_id'])
        require(len(ids)>=quotas[c], 'Insufficient category '+c+'; no automatic redistribution')
        chosen.extend(rng.choice(ids, size=quotas[c], replace=False).tolist())
    return frame.set_index('query_id',drop=False).loc[sorted(chosen)].reset_index(drop=True)

def runtime_contract():
    return {'python':platform.python_version(), 'packages':{n:importlib.metadata.version(n) for n in
        ('torch','transformers','tokenizers','huggingface-hub','numpy','pandas','Pillow')},
        'device':str(DEVICE), 'device_name':torch.cuda.get_device_name() if DEVICE.type=='cuda' else platform.machine(),
        'cuda':torch.version.cuda, 'dtype':'float32', 'amp':False, 'tf32':False, 'attention_backend':'eager'}

def seed_all(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

def rng_state():
    return {'python':random.getstate(),'numpy':np.random.get_state(),'torch':torch.get_rng_state(),
            'cuda':torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None}

def restore_rng(s):
    random.setstate(s['python']); np.random.set_state(s['numpy']); torch.set_rng_state(s['torch'])
    if s['cuda'] is not None:
        require(torch.cuda.is_available(), 'Resume CUDA checkpoint on CPU refused')
        torch.cuda.set_rng_state_all(s['cuda'])

def status(phase, **extra):
    write_json(RUN/'pilot_status.json', {'phase':phase,'implementation':IMPLEMENTATION,
        'official_val_used':False,'full_experiment_started':False,'updated_utc':pd.Timestamp.now(tz='UTC').isoformat(),**extra})

@contextlib.contextmanager
def stage(name):
    try: yield
    except BaseException as exc:
        if isinstance(exc, KeyboardInterrupt): kind='interrupted'
        else: kind='blocked'
        # No credential/download command logs are included in exception messages.
        write_bytes(RUN/(name+'_traceback.txt'), traceback.format_exc().encode())
        status(kind, stage=name, blocker=str(exc), traceback_file=name+'_traceback.txt')
        raise



In [ ]:
# checkpoint_architecture.py — code nhúng, không phụ thuộc file local
class ReferenceInjection(nn.Module):
    def __init__(self, text_dim, vision_dim, heads, mlp):
        super().__init__()
        self.q_ln=nn.LayerNorm(text_dim); self.v_ln=nn.LayerNorm(vision_dim)
        self.v_proj=nn.Linear(vision_dim,text_dim)
        self.ca=nn.MultiheadAttention(text_dim,heads,batch_first=True,dropout=0.0)
        self.out_ln=nn.LayerNorm(text_dim); self.mlp=copy.deepcopy(mlp)
    def forward(self, h, reference):
        v=self.v_proj(self.v_ln(reference))
        z=h+self.ca(self.q_ln(h),v,v,need_weights=False)[0]
        return z+self.mlp(self.out_ln(z))

def eos_positions(ids, mask, eos_token_id):
    require(ids.shape==mask.shape and bool(((mask==0)|(mask==1)).all()), 'Token/mask shape or values invalid')
    lengths=mask.sum(-1)
    require(bool((lengths>=2).all()), 'Empty text token sequence')
    expected=torch.arange(ids.shape[1],device=ids.device)[None,:]<lengths[:,None]
    require(torch.equal(expected,mask.bool()), 'Tokenizer must right-pad contiguous sequences')
    # pad_token == EOS for CLIP; intersect with attention_mask and select first real EOS.
    is_eos=(ids==eos_token_id)&mask.bool()
    require(bool(is_eos.any(-1).all()), 'EOS absent from unpadded tokens')
    pos=is_eos.long().argmax(-1)
    require(bool((pos==lengths-1).all()), 'EOS must close each untruncated modifier')
    return pos

def text_masks(mask, dtype):
    b,n=mask.shape; neg=torch.finfo(dtype).min
    padding=torch.zeros((b,1,n,n),device=mask.device,dtype=dtype).masked_fill(~mask[:,None,None,:].bool(),neg)
    causal=torch.full((n,n),neg,dtype=dtype,device=mask.device).triu(1)
    return padding,causal[None,None].expand(b,1,n,n)

class TMEPilot(nn.Module):
    """Paper-inspired layer-level tri-branch; not the unpublished original TME."""
    def __init__(self, base, eos_token_id, active_layers):
        super().__init__()
        self.text=copy.deepcopy(base.text_model)
        self.projection=copy.deepcopy(base.text_projection)
        self.text.requires_grad_(True);self.projection.requires_grad_(True)
        self.eos_token_id=int(eos_token_id); self.active_layers=tuple(active_layers)
        self.align=nn.ModuleDict(); self.inject=nn.ModuleDict(); self.merge=nn.ModuleDict()
        self.align_norm=nn.ModuleDict(); self.align_projection=nn.ModuleDict()
        self.text.embeddings.requires_grad_(False)  # pilot choice: freeze token/position tables
        tc=base.config.text_config; vc=base.config.vision_config
        require(len(self.text.encoder.layers)==vc.num_hidden_layers, 'This implementation requires paired text/vision depths')
        require(self.active_layers and len(set(self.active_layers))==len(self.active_layers) and
                all(0<=i<len(self.text.encoder.layers) for i in self.active_layers),'Invalid active layers')
        for i in self.active_layers:
            k=str(i); block=self.text.encoder.layers[i]
            self.align[k]=copy.deepcopy(block)
            self.inject[k]=ReferenceInjection(tc.hidden_size,vc.hidden_size,tc.num_attention_heads,block.mlp)
            self.merge[k]=nn.LayerNorm(tc.hidden_size)
            self.align_norm[k]=copy.deepcopy(self.text.final_layer_norm)
            self.align_projection[k]=copy.deepcopy(base.text_projection)
            self.align_projection[k].requires_grad_(True)

    def forward(self, reference, modifier, return_alignment=False, conventional_only=False):
        # reference: layer_tokens[B,L,Nv,Dv], pooled[B,D]; contains NO target.
        ids=modifier['input_ids']; mask=modifier['attention_mask']
        pos=eos_positions(ids,mask,self.eos_token_id); b=ids.shape[0]
        h=self.text.embeddings(input_ids=ids)
        padding,causal=text_masks(mask,h.dtype); aligned=[]
        valid=mask.unsqueeze(-1).to(h.dtype)
        for i,block in enumerate(self.text.encoder.layers):
            previous=h
            conventional=block(previous,attention_mask=padding,causal_attention_mask=causal)[0]
            if i in self.active_layers and not conventional_only:
                k=str(i)
                ri=self.inject[k](previous,reference['layer_tokens'][:,self.active_layers.index(i)])
                ta=self.align[k](previous,attention_mask=padding,causal_attention_mask=causal)[0]
                # Equation 4: enhanced stream is propagated into every subsequent layer.
                h=self.merge[k](conventional+ri+ta)*valid
                a=self.align_norm[k](ta)[torch.arange(b,device=h.device),pos]
                aligned.append(self.align_projection[k](a))
            else: h=conventional*valid
        h=self.text.final_layer_norm(h)
        pooled=h[torch.arange(b,device=h.device),pos]
        text=unit(self.projection(pooled))
        q=unit(text+unit(reference['pooled']))
        if return_alignment:
            require(aligned, 'No TA outputs in conventional-only path')
            return q, torch.stack(aligned,dim=1)
        return q

    def encode_query(self, reference, modifier):
        return self(reference, modifier)  # deliberately no target argument



In [ ]:
# data_base.py — code nhúng, không phụ thuộc file local
def init_trial():
    global RUN,DEVICE,LOCAL
    require(CFG['seed']==42 and CFG['subset_size']==2048 and 1<=CFG['max_epochs']<=3,'Trial scope: seed42, 2048, <=3 epochs')
    require(CFG['amp'] is False and CFG['tf32'] is False,'Only verified FP32 trial; no automatic AMP/TF32')
    require(CFG['batch_size'] in (32,64),'Choose 32/64 before locking experiment')
    require(Path(CFG['experiment_id']).name==CFG['experiment_id'],'Invalid experiment ID')
    DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    require(not CFG['require_cuda'] or DEVICE.type=='cuda','Select Colab GPU before proceeding')
    torch.backends.cuda.matmul.allow_tf32=False;torch.backends.cudnn.allow_tf32=False;torch.backends.cudnn.benchmark=False
    torch.set_default_dtype(torch.float32)
    seed_all(42);RUN=Path(CFG['output_root'])/CFG['experiment_id'];LOCAL=Path(CFG['local_root'])/CFG['experiment_id']
    require(not RUN.is_relative_to(Path(CFG['development_root'])) and not RUN.is_relative_to(Path(CFG['preparation_root'])),'Output overlaps source')
    if RUN.exists() and any(RUN.iterdir()):require(CFG['resume'],'Existing run: resume=True or new experiment ID')
    RUN.mkdir(parents=True,exist_ok=True);LOCAL.mkdir(parents=True,exist_ok=True)
    write_json(RUN/'trial_request.json',{'config':CFG,'code_sha256':CODE_SHA256,'implementation':'visfit-residual-adapter-v1'},immutable=True)
    write_json(RUN/'bootstrap_report.json',BOOTSTRAP)

def inspect_sources():
    prep=Path(CFG['preparation_root']);dev=Path(CFG['development_root'])
    for name,sha in PREPARATION_ANCHORS.items():require(file_hash(prep/name)==sha,'Preparation changed: '+name)
    for name,sha in SOURCE_LOCKS.items():
        root=dev/'conventional' if name in ('config.json','best.json','metric_history.json','epoch_03_per_query.csv') else dev
        require(file_hash(root/name)==sha,'Conventional provenance changed: '+name)
    contract=read_json(prep/'extraction_contract.json');meta=read_json(prep/'feature_metadata.json')
    require(meta['contract']==contract and meta['status']=='complete' and not meta['legacy_arrays_mixed'],'Incomplete/mixed preparation')
    require(contract['caption_policy']==CAPTION_POLICY and contract['dtype']=='float32' and not contract['autocast'] and not contract['tf32'],'Extraction/caption mismatch')
    require(file_hash(prep/'feature_audit.json')==meta['feature_audit_sha256'],'Preparation feature audit checksum')
    dev_manifests=read_json(dev/'pilot_manifest_hashes.json')
    require(dev_manifests['binding']['feature_fingerprint']==meta['fingerprint'] and dev_manifests['binding']['preparation_anchors']==PREPARATION_ANCHORS,'Conventional was trained with different preparation')
    data_audit=read_json(prep/'data_audit.json');require(data_audit.get('schema_error_count')==0 and data_audit.get('both_empty_captions_count')==0 and not data_audit.get('query_or_gallery_errors'),'Preparation audit did not pass')
    frames={};warnings=[];plan=read_json(prep/'training_plan.json')
    for name in ('development_train','development_val','development_gallery'):
        f=pd.read_csv(prep/(name+'.csv'),dtype=str,keep_default_na=False)
        vr=meta['feature_views'][name];view=pd.read_csv(prep/vr['path'],dtype=str,keep_default_na=False)
        require(file_hash(prep/vr['path'])==vr['sha256'] and f.equals(view[f.columns]) and len(f)==vr['rows'],'Feature view mapping changed')
        if 'query_id' in f:
            require(f.query_id.is_unique and set(f.official_split)=={'train'},'Only official-train derived development queries')
            for row in f.itertuples():
                require(row.source_file==f'cap.{row.category}.train.json' and row.query_id==f'{row.category}:train:{int(row.source_row):06d}','Query/source mismatch')
                error,warning=assess_captions([row.caption_1,row.caption_2]);require(error is None,'Caption schema/content: '+row.query_id)
                require(row.modification_text==caption_text(row.caption_1,row.caption_2),'Caption policy changed')
                require(row.triplet_key_sha256==fingerprint([row.category,row.reference_image_id,row.target_image_id,row.caption_1,row.caption_2]),'Triplet identity mismatch')
                if warning:warnings.append({'query_id':row.query_id,'reason':warning})
        else:
            f.gallery_index=f.gallery_index.astype(int)
            for c in CATEGORIES:
                g=f[f.category==c];require(g.image_id.is_unique and g.gallery_index.tolist()==list(range(len(g))),'Gallery order mismatch')
                require(fingerprint(g.image_id.tolist())==plan['development']['gallery_id_order_sha256'][c],'Gallery lock changed')
        frames[name]=f
    train,val,gallery=[frames[k] for k in ('development_train','development_val','development_gallery')]
    require(len(train)==16204 and len(val)==1796 and len(gallery)==1796,'Locked counts changed')
    require(set(train.query_id).isdisjoint(val.query_id) and set(train.target_image_id).isdisjoint(val.target_image_id),'Development leakage')
    for c in CATEGORIES:require(set(val[val.category==c].target_image_id)<=set(gallery[gallery.category==c].image_id),'Validation target coverage')
    subset=stratified_sample(train,CFG['subset_size'],42)
    for name,f in [('subset',subset),('validation',val),('gallery',gallery)]:write_csv(RUN/'manifests'/(name+'.csv'),f,immutable=True)
    require(file_hash(RUN/'manifests'/'validation.csv')==dev_manifests['manifests']['pilot_val']['sha256'] and file_hash(RUN/'manifests'/'gallery.csv')==dev_manifests['manifests']['development_gallery']['sha256'],'Validation/gallery differ from conventional selection protocol')
    write_json(RUN/'caption_warnings.json',warnings)
    source={'prep':prep,'dev':dev,'meta':meta,'contract':contract,'subset':subset,'validation':val,'gallery':gallery,
        'subset_sha256':file_hash(RUN/'manifests'/'subset.csv'),'validation_sha256':file_hash(RUN/'manifests'/'validation.csv'),
        'gallery_sha256':file_hash(RUN/'manifests'/'gallery.csv'),'mapping_hashes':dev_manifests['binding']['mapping_hashes']}
    write_json(RUN/'source_provenance.json',{'source_locks':SOURCE_LOCKS,'prep_anchors':PREPARATION_ANCHORS,'base_marker':EXPECTED_BASE_MARKER,
        'subset_sha256':source['subset_sha256'],'feature_fingerprint':meta['fingerprint'],'counts':{k:len(source[k]) for k in ('subset','validation','gallery')}},immutable=True)
    return source

class FrozenConventional(nn.Module):
    def __init__(self,text,projection,eos_id):
        super().__init__();self.text=text;self.projection=projection;self.eos_id=eos_id;self.requires_grad_(False);self.eval()
    def train(self,mode=True):return super().train(False)
    def encode_text(self,modifier):
        ids,mask=modifier['input_ids'],modifier['attention_mask'];eos=eos_positions(ids,mask,self.eos_id)
        h=self.text.embeddings(input_ids=ids);padding,causal=text_masks(mask,h.dtype);valid=mask.unsqueeze(-1).to(h.dtype)
        for block in self.text.encoder.layers:h=block(h,attention_mask=padding,causal_attention_mask=causal)[0]*valid
        return self.text.final_layer_norm(h)*valid,mask,eos
    def encode_query(self,reference,modifier):
        h,mask,eos=self.encode_text(modifier);v=self.projection(h[torch.arange(len(h),device=h.device),eos])
        return unit(unit(v)+unit(reference['pooled']))

def load_frozen_checkpoint(source):
    c=source['contract'];dev=source['dev'];record=read_json(dev/'conventional'/'config.json');marker=read_json(dev/'conventional'/'best.json')
    require(marker==EXPECTED_BASE_MARKER and marker['phase']=='conventional' and record['variant']=='conventional','Wrong base checkpoint')
    path=dev/'conventional'/'best.pt';require(file_hash(path)==marker['sha256'],'Base checkpoint SHA mismatch')
    require(c['encoder_id']=='patrickjohncyh/fashion-clip' and c['revision']=='7e3ba62ce16b379a1ab479346b66f192e76f51b7','Unexpected backbone revision')
    tokenizer=CLIPTokenizer.from_pretrained(c['encoder_id'],revision=c['revision'])
    processor=CLIPImageProcessor.from_pretrained(c['encoder_id'],revision=c['revision'])
    require(fingerprint(tokenizer.get_vocab())==c['tokenizer']['vocab_sha256'] and tokenizer.special_tokens_map==c['tokenizer']['special_tokens'] and tokenizer.padding_side=='right','Tokenizer mismatch')
    require(all(processor.to_dict().get(k)==v for k,v in c['processor'].items() if k not in ('_processor_class','image_processor_type')),'Preprocessor mismatch')
    base=CLIPModel.from_pretrained(c['encoder_id'],revision=c['revision'],dtype=torch.float32,attn_implementation='eager')
    require(tensor_hash(base)==c['encoder_weight_sha256'],'Pretrained weight hash mismatch');base.requires_grad_(False);base.eval();base.to(DEVICE)
    state=torch.load(path,map_location='cpu',weights_only=False) # own hash-bound stored checkpoint; never arbitrary pickle
    required={'signature','phase','step','epoch','cursor','model','optimizer','rng','config','cache_fingerprint','metrics','best_epoch','finished'}
    require(required<=state.keys() and state['signature']==record['signature']==marker['signature'] and state['phase']=='conventional','Checkpoint payload schema/signature mismatch')
    require(state['step']==12153 and state['epoch']==3 and state['best_epoch']==3 and state['finished'] and state['config']==record['config'],'Checkpoint selection/config changed')
    require(state['cache_fingerprint']==read_json(dev/'cache_binding.json')['fingerprint'],'Development cache provenance mismatch')
    wrapper=TMEPilot(base,tokenizer.eos_token_id,list(range(base.config.text_config.num_hidden_layers)))
    wrapper.load_state_dict(state['model'],strict=True);wrapper.requires_grad_(False);wrapper.eval();wrapper.to(DEVICE)
    frozen=FrozenConventional(wrapper.text,wrapper.projection,tokenizer.eos_token_id).to(DEVICE)
    # Small genuine encoder forward and strict schema parity, before larger feature reads.
    saved=rng_state()
    try:
        mod=tokenizer(['Red and longer',' and blue'],padding='max_length',max_length=77,return_tensors='pt');mod={k:v.to(DEVICE) for k,v in mod.items()}
        image=Image.fromarray(np.arange(224*224*3,dtype=np.uint32).reshape(224,224,3).astype(np.uint8))
        pixels=processor(images=[image,image],return_tensors='pt')['pixel_values'].to(DEVICE,dtype=torch.float32)
        with torch.inference_mode():
            out=base.vision_model(pixel_values=pixels);ref={'pooled':unit(base.visual_projection(out.pooler_output))}
            q0=wrapper(ref,mod,conventional_only=True);q1=frozen.encode_query(ref,mod)
            native=unit(unit(frozen.projection(frozen.text(**mod).pooler_output))+unit(ref['pooled']))
            require(torch.equal(q0,q1),'Extracted conventional differs from strict checkpoint architecture')
            require(torch.allclose(q1,native,atol=3e-5,rtol=3e-5),'Native trained text/adapter parity failed')
            finite(out.last_hidden_state,'vision');finite(q1,'query');require(q1.shape==(2,512),'Query shape')
    finally:restore_rng(saved)
    hashes={'vision':tensor_hash(base.vision_model),'visual_projection':tensor_hash(base.visual_projection),'text':tensor_hash(frozen)}
    write_json(RUN/'base_preflight.json',{'passed':True,'strict_load':True,'checkpoint_sha256':marker['sha256'],'schema_keys':sorted(state.keys()),
        'extracted_conventional_bitwise_parity':True,'native_forward_parity':True,'hashes':hashes,'checkpoint_epoch':state['epoch'],'device':str(DEVICE),
        'module_paths':{k:getattr(sys.modules.get(k),'__file__',None) for k in ('torch','torchvision','transformers','tokenizers','huggingface_hub')},
        'packages':{k:importlib.metadata.version(k) for k in ('torch','torchvision','transformers','tokenizers','huggingface-hub','numpy','pandas','Pillow')}})
    del state,wrapper
    return base,frozen,tokenizer,processor,hashes

def assert_backbones_frozen(base,frozen,hashes,check_hash=True):
    require(not base.training and not frozen.training and all(not p.requires_grad and p.grad is None for m in (base,frozen) for p in m.parameters()),'Frozen backbone mode/grad violation')
    if check_hash:require({'vision':tensor_hash(base.vision_model),'visual_projection':tensor_hash(base.visual_projection),'text':tensor_hash(frozen)}==hashes,'Frozen backbone weights changed')


In [ ]:
# cache.py — code nhúng, không phụ thuộc file local
class PreparationStore:
    def __init__(self,source,kind):
        require(kind in ('reference','target'),'Only final image features may be reused')
        self.kind=kind;self.root=source['prep']/'features'/source['meta']['fingerprint']/kind
        require(file_hash(self.root/'index.json')==source['meta']['store_index_sha256'][kind],'Source feature index SHA')
        self.index=read_json(self.root/'index.json');self.ids=self.index['ids'];self.mapping={v:i for i,v in enumerate(self.ids)}
        require(len(self.mapping)==len(self.ids) and self.index['fingerprint']==source['meta']['fingerprint'] and self.index['kind']==kind,'Source index identity')
        mapping_path=self.root.parent/source['meta']['mapping_files'][kind]
        require(file_hash(mapping_path)==source['mapping_hashes'][kind] and read_json(mapping_path)==self.mapping,'Source mapping differs from conventional')
        require(self.index['shape_tail']==({'reference':[50,768],'target':[512]}[kind]) and self.index['dtype']=='float32','Image feature schema')
        self.local=LOCAL/'source_images'/source['meta']['fingerprint']/kind;self.local.mkdir(parents=True,exist_ok=True)
        self.verified=set();self.loaded=OrderedDict()
    def take(self,ids):
        result=[];step=self.index['shard_size']
        for image_id in ids:
            require(image_id in self.mapping,'Image ID absent: '+image_id)
            row=self.mapping[image_id];start=row//step*step;end=min(start+step,len(self.ids));name=f'{start:07d}_{end:07d}'
            local=self.local/(name+'.npy')
            if start not in self.verified:
                marker=read_json(self.root/(name+'.json'))
                expected={'fingerprint':self.index['fingerprint'],'kind':self.kind,'ids_sha256':fingerprint(self.ids[start:end]),'shape':[end-start,*self.index['shape_tail']],'dtype':'float32'}
                require(all(marker.get(k)==v for k,v in expected.items()),'Source shard schema/ID mismatch')
                if not local.is_file() or file_hash(local)!=marker['sha256']:
                    temp=local.with_suffix('.copying');shutil.copyfile(self.root/(name+'.npy'),temp)
                    require(file_hash(temp)==marker['sha256'],'Source feature corrupt');os.replace(temp,local)
                a=np.load(local,mmap_mode='r',allow_pickle=False)
                require(list(a.shape)==expected['shape'] and a.dtype==np.float32 and np.isfinite(a).all(),'Source contents invalid')
                if self.kind=='target':require(np.allclose(np.linalg.norm(a,axis=-1),1,atol=1e-5,rtol=1e-5),'Source target not unit')
                self.verified.add(start)
            if start not in self.loaded:
                if len(self.loaded)>=16:self.loaded.popitem(last=False)
                self.loaded[start]=np.load(local,mmap_mode='r',allow_pickle=False)
            self.loaded.move_to_end(start);result.append(np.asarray(self.loaded[start][row-start]).copy())
        return np.stack(result)

class TrialStore:
    def __init__(self,root,local,contract,specs,shard_size):
        self.key=fingerprint(contract);self.root=Path(root)/self.key;self.local=Path(local)/self.key
        self.root.mkdir(parents=True,exist_ok=True);self.local.mkdir(parents=True,exist_ok=True)
        self.specs=specs;self.contract=contract;self.step=shard_size;self.maps={k:{v:i for i,v in enumerate(s['ids'])} for k,s in specs.items()}
        require(all(len(self.maps[k])==len(s['ids']) for k,s in specs.items()),'Duplicate cached IDs')
        self.index={'fingerprint':self.key,'contract':contract,'specs':specs,'shard_size':shard_size}
        write_json(self.root/'index.json',self.index,immutable=True);self.loaded=OrderedDict();self.paths={}
    def prepare(self,role,compute):
        spec=self.specs[role];ids=spec['ids'];dtype=np.dtype(spec['dtype']);tail=spec['shape']
        (self.root/role).mkdir(exist_ok=True);(self.local/role).mkdir(exist_ok=True)
        for start in range(0,len(ids),self.step):
            end=min(start+self.step,len(ids));name=f'{start:07d}_{end:07d}';p=self.root/role/(name+'.npy');marker=p.with_suffix('.json');pending=p.with_suffix('.pending.json')
            expected={'fingerprint':self.key,'role':role,'ids_sha256':fingerprint(ids[start:end]),'shape':[end-start,*tail],'dtype':spec['dtype']}
            if not marker.is_file() and p.is_file() and pending.is_file():
                m=read_json(pending);require(all(m.get(k)==v for k,v in expected.items()) and file_hash(p)==m['sha256'],'Pending cache corrupt');write_json(marker,m,immutable=True)
            local=self.local/role/p.name
            if marker.is_file():
                m=read_json(marker);require(all(m.get(k)==v for k,v in expected.items()),'Cache fingerprint/ID/shape differs')
                require(p.is_file() and file_hash(p)==m['sha256'],'Committed Drive cache corrupt')
                if not local.is_file() or file_hash(local)!=m['sha256']:
                    temp=local.with_suffix('.copying');shutil.copyfile(p,temp);require(file_hash(temp)==m['sha256'],'Local cache copy corrupt');os.replace(temp,local)
            else:
                require(not p.exists(),'Uncommitted orphan cache; preserve and investigate')
                a=np.asarray(compute(ids[start:end]),dtype=dtype)
                require(list(a.shape)==expected['shape'] and np.isfinite(a).all(),'Computed cache shape/finite')
                temp=local.with_suffix('.partial');
                with temp.open('wb') as f:np.save(f,a,allow_pickle=False)
                m={**expected,'sha256':file_hash(temp)};write_json(pending,m)
                remote=p.with_suffix('.copying');shutil.copyfile(temp,remote);require(file_hash(remote)==m['sha256'],'Cache sync corrupt')
                os.replace(remote,p);write_json(marker,m,immutable=True);os.replace(temp,local)
            a=np.load(local,mmap_mode='r',allow_pickle=False)
            require(list(a.shape)==expected['shape'] and a.dtype==dtype and np.isfinite(a).all(),'Local cache invalid')
            if role in ('reference_pooled','target_pooled'):require(np.allclose(np.linalg.norm(a,axis=-1),1,atol=1e-5,rtol=1e-5),'Cached embedding not unit')
            self.paths[role,start]=local
    def take(self,role,ids):
        # Training hot path: prevalidated local mmap, no Drive/JSON/checksum access.
        result=[]
        for image_id in ids:
            row=self.maps[role][image_id];start=row//self.step*self.step;key=(role,start)
            require(key in self.paths,'Run cache preflight before use')
            if key not in self.loaded:
                if len(self.loaded)>=16:self.loaded.popitem(last=False)
                self.loaded[key]=np.load(self.paths[key],mmap_mode='r',allow_pickle=False)
            self.loaded.move_to_end(key);result.append(np.asarray(self.loaded[key][row-start]).copy())
        return np.stack(result)

def prepare_cache(source,base,frozen,tokenizer):
    ref_store=PreparationStore(source,'reference');target_store=PreparationStore(source,'target')
    for role in ('subset','validation'):
        view_name='development_train' if role=='subset' else 'development_val'
        vr=source['meta']['feature_views'][view_name];view=pd.read_csv(source['prep']/vr['path'],dtype=str,keep_default_na=False).set_index('query_id',drop=False)
        for row in source[role].itertuples():
            v=view.loc[row.query_id]
            require(int(v.reference_feature_index)==ref_store.mapping[row.reference_image_id] and int(v.target_feature_index)==target_store.mapping[row.target_image_id],'Source row/ID alignment differs')
    queries=pd.concat([source['subset'],source['validation']],ignore_index=True).sort_values('query_id').reset_index(drop=True)
    require(queries.query_id.is_unique,'Train/val query overlap');query_ids=queries.query_id.tolist();by_id=queries.set_index('query_id',drop=False)
    refs=sorted(set(queries.reference_image_id));targets=sorted(set(source['subset'].target_image_id)|set(source['gallery'].image_id))
    contract={'implementation':'visfit-adapter-cache-v1','code_sha256':CODE_SHA256,'base_checkpoint_sha256':EXPECTED_BASE_MARKER['sha256'],
        'text':'strict trained conventional last stream -> final LN, padded outputs zero; frozen trained projection applied later',
        'image_extraction_contract':source['contract'],'feature_fingerprint':source['meta']['fingerprint'],
        'subset_sha256':source['subset_sha256'],'validation_sha256':source['validation_sha256'],'gallery_sha256':source['gallery_sha256'],
        'source_image_index_sha256':{k:source['meta']['store_index_sha256'][k] for k in ('reference','target')},
        'text_encode_batch_size':CFG['encode_batch_size'],'cache_shard_size':CFG['cache_shard_size'],
        'caption_policy':CAPTION_POLICY,'dtype':'float32','attention_backend':'eager','amp':False,'tf32':False,
        'runtime':{'packages':{k:importlib.metadata.version(k) for k in ('torch','transformers','numpy')},'device':str(DEVICE),'gpu':torch.cuda.get_device_name() if DEVICE.type=='cuda' else None}}
    specs={'text':{'ids':query_ids,'shape':[77,512],'dtype':'float32'},'mask':{'ids':query_ids,'shape':[77],'dtype':'int64'},
        'eos':{'ids':query_ids,'shape':[],'dtype':'int64'},'input_ids':{'ids':query_ids,'shape':[77],'dtype':'int64'},
        'reference':{'ids':refs,'shape':[50,768],'dtype':'float32'},'reference_pooled':{'ids':refs,'shape':[512],'dtype':'float32'},
        'target_pooled':{'ids':targets,'shape':[512],'dtype':'float32'}}
    payload=sum(len(s['ids'])*math.prod(s['shape'])*np.dtype(s['dtype']).itemsize for s in specs.values())
    # Source final-image shards are small; reserve their full image-store footprint plus checkpoint/sync temporaries.
    source_bytes=sum(len(st.ids)*math.prod(st.index['shape_tail'])*4 for st in (ref_store,target_store))
    write_json(RUN/'cache_resource_plan.json',{'adapter_cache_payload_bytes':payload,'source_image_store_upper_bound_bytes':source_bytes,
        'local_disk_free':shutil.disk_usage(LOCAL).free,'no_multilayer_cache':True})
    key=fingerprint(contract);existing=sum(p.stat().st_size for p in (LOCAL/'cache'/key).rglob('*.npy')) if (LOCAL/'cache'/key).exists() else 0
    require(shutil.disk_usage(LOCAL).free>max(0,payload-existing)+source_bytes+3*1024**3,'Insufficient local disk before cache')
    store=TrialStore(RUN/'cache',LOCAL/'cache',contract,specs,CFG['cache_shard_size'])
    def text_values(ids,role):
        rows=by_id.loc[ids];values=[]
        for start in range(0,len(rows),CFG['encode_batch_size']):
            r=rows.iloc[start:start+CFG['encode_batch_size']];sentences=r.modification_text.tolist()
            lengths=[len(x) for x in tokenizer(sentences,padding=False,truncation=False)['input_ids']]
            require(max(lengths)<=77,'Overlength caption; do not truncate/drop')
            mod=tokenizer(sentences,padding='max_length',max_length=77,truncation=False,return_tensors='pt');mod={k:v.to(DEVICE) for k,v in mod.items()}
            if role=='text':
                with torch.inference_mode():value=frozen.encode_text(mod)[0]
            elif role=='eos':value=eos_positions(mod['input_ids'],mod['attention_mask'],tokenizer.eos_token_id)
            else:value=mod['attention_mask' if role=='mask' else 'input_ids']
            values.append(value.cpu().numpy())
        return np.concatenate(values)
    # Only missing text shards call the frozen encoder; mask/EOS/IDs need tokenizer alone.
    for role in ('text','mask','eos','input_ids'):store.prepare(role,lambda ids,r=role:text_values(ids,r))
    store.prepare('reference',ref_store.take)
    def pooled_refs(ids):
        values=[]
        for start in range(0,len(ids),CFG['encode_batch_size']):
            raw=torch.from_numpy(ref_store.take(ids[start:start+CFG['encode_batch_size']])).to(DEVICE)
            with torch.inference_mode():p=unit(base.visual_projection(base.vision_model.post_layernorm(raw[:,0])))
            values.append(p.cpu().numpy())
        return np.concatenate(values)
    store.prepare('reference_pooled',pooled_refs);store.prepare('target_pooled',target_store.take)
    # Exhaustive alignment checks on every used query; this is not a sample audit.
    for start in range(0,len(query_ids),128):
        ids=query_ids[start:start+128];mask=torch.from_numpy(store.take('mask',ids));token_ids=torch.from_numpy(store.take('input_ids',ids));eos=torch.from_numpy(store.take('eos',ids))
        require(torch.equal(eos_positions(token_ids,mask,tokenizer.eos_token_id),eos),'Cached mask/EOS mapping')
        seq=store.take('text',ids);require(np.count_nonzero(seq[mask.numpy()==0])==0,'Nonzero cached padding')
    write_json(RUN/'cache_audit.json',{'passed':True,'fingerprint':store.key,'queries_verified':len(query_ids),'source_roles':['reference','target'],
        'text_from_trained_checkpoint':True,'all_used_shards_checksum_finite_shape_verified':True,'multilayer_cache_read':False})
    return store

def make_batch(rows,store,training=True):
    ids=rows.query_id.tolist();refids=rows.reference_image_id.tolist()
    batch={'reference_features':{'tokens':torch.from_numpy(store.take('reference',refids)).to(DEVICE),'pooled':torch.from_numpy(store.take('reference_pooled',refids)).to(DEVICE)},
        'text_features':torch.from_numpy(store.take('text',ids)).to(DEVICE),'mask':torch.from_numpy(store.take('mask',ids)).to(DEVICE),
        'eos':torch.from_numpy(store.take('eos',ids)).to(DEVICE),'query_ids':ids}
    if training:batch.update(target=torch.from_numpy(store.take('target_pooled',rows.target_image_id.tolist())).to(DEVICE),target_ids=rows.target_image_id.tolist())
    return batch


In [ ]:
# adapter.py — code nhúng, không phụ thuộc file local
class ResidualAdapter(nn.Module):
    def __init__(self,projection_weight,text_dim=512,vision_dim=768,width=128,heads=4):
        super().__init__();require(width%heads==0,'Adapter width/head mismatch')
        self.register_buffer('projection_weight',projection_weight.detach().clone())
        self.text_norm=nn.LayerNorm(text_dim);self.text_down=nn.Linear(text_dim,width)
        self.reference_norm=nn.LayerNorm(vision_dim);self.reference_down=nn.Linear(vision_dim,width)
        self.cross_attention=nn.MultiheadAttention(width,heads,dropout=0.,batch_first=True)
        self.ri_mlp=nn.Sequential(nn.Linear(width,width),nn.GELU(),nn.Linear(width,text_dim))
        self.ta_mlp=nn.Sequential(nn.Linear(text_dim,width),nn.GELU(),nn.Linear(width,text_dim))
        self.gate_ri=nn.Parameter(torch.zeros(()));self.gate_ta=nn.Parameter(torch.zeros(()))
    def forward(self,reference_features,text_features,mask,eos,return_aux=False,gates_off=False):
        require(text_features.ndim==3 and mask.shape==text_features.shape[:2] and eos.shape==(len(text_features),),'Adapter input shape')
        require(bool(((mask==0)|(mask==1)).all()) and bool((eos==mask.sum(-1)-1).all()),'Mask/EOS invalid')
        require(bool((eos>=1).all()) and bool((mask.sum(-1)<=mask.shape[1]).all()),'Invalid sequence length')
        require(torch.equal(mask.bool(),torch.arange(mask.shape[1],device=mask.device)[None,:]<mask.sum(-1)[:,None]),'Mask must be contiguous right padding')
        valid=mask.unsqueeze(-1).to(text_features.dtype)
        x=self.text_norm(text_features);q=self.text_down(x);kv=self.reference_down(self.reference_norm(reference_features['tokens']))
        ri=self.cross_attention(q,kv,kv,need_weights=False)[0];ri=self.ri_mlp(ri)*valid
        ta=self.ta_mlp(x)*valid
        gri=self.gate_ri.new_zeros(()) if gates_off else torch.tanh(self.gate_ri)
        gta=self.gate_ta.new_zeros(()) if gates_off else torch.tanh(self.gate_ta)
        enhanced=(text_features+gri*ri+gta*ta)*valid # no LN/MLP on the base stream
        index=torch.arange(len(eos),device=eos.device)
        text=unit(F.linear(enhanced[index,eos],self.projection_weight))
        query=unit(text+unit(reference_features['pooled'])) # reference composed exactly once
        if return_aux:
            student=F.linear((text_features+ta)[index,eos],self.projection_weight)
            return query,student,enhanced
        return query
    def encode_query(self,reference_features,text_features,mask,eos):
        return self(reference_features,text_features,mask,eos)
    def enhanced_text_sequence(self,reference_features,text_features,mask,eos):
        _,_,h=self(reference_features,text_features,mask,eos,return_aux=True)
        return h,mask,eos

def positive_mask(target_ids,device):
    require(target_ids and all(isinstance(v,str) and v for v in target_ids),'Invalid target IDs')
    return torch.tensor([[a==b for b in target_ids] for a in target_ids],dtype=torch.bool,device=device)

def retrieval_loss(query,target,target_ids,temperature):
    require(temperature>0 and query.shape==target.shape,'InfoNCE shape/temperature')
    logits=unit(query)@unit(target.detach()).T/temperature;p=positive_mask(target_ids,query.device)
    def direction(x,m):return -((F.log_softmax(x,dim=1)*m).sum(1)/m.sum(1)).mean()
    return (direction(logits,p)+direction(logits.T,p.T))/2

def cosine_alignment(student,target):
    require(student.shape==target.shape and student.ndim==2,'Cosine alignment shape')
    return (1-(unit(student)*unit(target.detach())).sum(-1)).mean()

def objective(model,batch,weight):
    q,student,_=model(batch['reference_features'],batch['text_features'],batch['mask'],batch['eos'],return_aux=True)
    retrieval=retrieval_loss(q,batch['target'],batch['target_ids'],CFG['retrieval_temperature'])
    alignment=cosine_alignment(student,batch['target']);total=retrieval+weight*alignment
    finite(total,'adapter loss');return total,retrieval,alignment

def adapter_initial(projection_weight):
    saved=rng_state();seed_all(42)
    try:return ResidualAdapter(projection_weight,text_dim=projection_weight.shape[1],vision_dim=CFG.get('vision_dim',768),width=CFG['adapter_width'],heads=CFG['adapter_heads']).float().to(DEVICE)
    finally:restore_rng(saved)

def gradient_summary(model):
    groups={'gate':[model.gate_ri,model.gate_ta],
        'RI':list(model.text_down.parameters())+list(model.reference_norm.parameters())+list(model.reference_down.parameters())+list(model.cross_attention.parameters())+list(model.ri_mlp.parameters()),
        'TA':list(model.ta_mlp.parameters()),'shared_norm':list(model.text_norm.parameters())}
    out=[]
    for params in groups.values():out.append(torch.stack([p.grad.detach().float().square().sum() for p in params if p.grad is not None]).sum().sqrt() if any(p.grad is not None for p in params) else model.gate_ri.new_zeros(()))
    values=torch.stack(out).detach().cpu().tolist();require(all(math.isfinite(x) for x in values),'Nonfinite branch gradient')
    return dict(zip(groups,values))

def alignment_influence(model,batch,weight):
    _,r,a=objective(model,batch,weight);params=list(model.parameters());gr=torch.autograd.grad(r,params,retain_graph=True,allow_unused=True)
    ga=torch.autograd.grad(weight*a,params,allow_unused=True) if weight else [None]*len(params)
    zero=r.new_zeros(());nr=sum((g.detach().float().square().sum() for g in gr if g is not None),zero)
    na=sum((g.detach().float().square().sum() for g in ga if g is not None),zero);dot=sum(((x.detach().float()*y.detach().float()).sum() for x,y in zip(gr,ga) if x is not None and y is not None),zero)
    v=torch.stack([nr.sqrt(),na.sqrt(),dot/(nr*na).sqrt().clamp_min(1e-30)]).cpu().tolist()
    return dict(zip(('retrieval_gradient_norm','weighted_alignment_gradient_norm','gradient_cosine'),v))

@torch.inference_mode()
def evaluate_adapter(model,source,store,baseline=False):
    model.eval();rows=[];per={};queries=source['validation'];gallery=source['gallery']
    for category in CATEGORIES:
        frame=queries[queries.category==category];g=gallery[gallery.category==category].sort_values('gallery_index',kind='stable')
        require(g.gallery_index.tolist()==list(range(len(g))) and g.image_id.is_unique,'Gallery alignment')
        lookup={v:i for i,v in enumerate(g.image_id)};vectors=store.take('target_pooled',g.image_id.tolist());ranks=[]
        for start in range(0,len(frame),CFG['eval_batch_size']):
            f=frame.iloc[start:start+CFG['eval_batch_size']];b=make_batch(f,store,training=False)
            require('target' not in b,'Target leaked into inference')
            q=model(b['reference_features'],b['text_features'],b['mask'],b['eos'],gates_off=baseline).cpu().numpy()
            r=exact_ranks(q,vectors,[lookup[v] for v in f.target_image_id]);ranks.extend(r.tolist())
            rows.extend({'query_id':v.query_id,'category':category,'reference_image_id':v.reference_image_id,'target_image_id':v.target_image_id,'target_rank':int(rank)} for v,rank in zip(f.itertuples(),r))
        per[category]={f'R@{k}':float((np.asarray(ranks)<=k).mean()*100) for k in (1,5,10,50)}
    require(len(rows)==len(queries) and {r['query_id'] for r in rows}==set(queries.query_id),'Missing validation queries')
    return {'scope':'fixed full development validation only','n_query':len(rows),'per_category':per,**macro_metrics(per),
       'query_counts':queries.groupby('category').size().to_dict(),'gallery_counts':gallery.groupby('category').size().to_dict(),'unit':'percent'},pd.DataFrame(rows)

def verify_baseline(source,store,frozen):
    model=adapter_initial(frozen.projection.weight);start=time.perf_counter();metric,pred=evaluate_adapter(model,source,store,baseline=True)
    expected=EXPECTED_METRICS;deviations={'Mean':metric['Mean']-expected['Mean']}
    for c in CATEGORIES:
        for k in (1,5,10,50):deviations[c+f'/R@{k}']=metric['per_category'][c][f'R@{k}']-expected['per_category'][c][f'R@{k}']
    old=pd.read_csv(source['dev']/'conventional'/'epoch_03_per_query.csv',dtype={'query_id':str})
    merged=pred.merge(old,on=['query_id','category','reference_image_id','target_image_id'],suffixes=('_new','_saved'),validate='one_to_one')
    require(len(merged)==len(pred)==1796,'Baseline prediction mapping mismatch')
    write_csv(RUN/'baseline'/'per_query.csv',pred);write_csv(RUN/'baseline'/'rank_comparison.csv',merged)
    passed=max(abs(v) for v in deviations.values())<=CFG['baseline_max_metric_deviation_pp']
    report={'passed':passed,'metrics':metric,'saved_metrics':expected,'deviation_pp':deviations,'tolerance_pp':CFG['baseline_max_metric_deviation_pp'],
      'rank_difference_count':int((merged.target_rank_new!=merged.target_rank_saved).sum()),'elapsed_seconds':time.perf_counter()-start,'trainable_parameters':0,
      'checkpoint_sha256':EXPECTED_BASE_MARKER['sha256'],'diagnosis_if_failed':'STOP: check strict checkpoint/epoch, captions, reference CLS->postLN/projection, trained text EOS/projection, ID mapping and stable exact ranking. Do not increase tolerance or switch checkpoint.'}
    write_json(RUN/'baseline'/'verification.json',report)
    require(passed,'Baseline reproduction blocked; inspect baseline/verification.json and rank_comparison.csv before training')
    return report


In [ ]:
# ranking.py — code nhúng, không phụ thuộc file local
def exact_ranks(query,gallery,target_positions):
    q=np.asarray(query,dtype=np.float32); g=np.asarray(gallery,dtype=np.float32)
    require(q.ndim==g.ndim==2 and q.shape[1]==g.shape[1] and len(q)>0 and len(g)>0,'Ranking shapes invalid')
    require(np.isfinite(q).all() and np.isfinite(g).all(),'Nonfinite ranking input')
    require((np.linalg.norm(q,axis=-1)>1e-8).all() and (np.linalg.norm(g,axis=-1)>1e-8).all(),'Zero ranking input')
    q=q/np.linalg.norm(q,axis=-1,keepdims=True);g=g/np.linalg.norm(g,axis=-1,keepdims=True)
    targets=np.asarray(target_positions,dtype=np.int64)
    require(targets.shape==(len(q),) and (targets>=0).all() and (targets<len(g)).all(),'Target gallery alignment invalid')
    scores=q@g.T; ts=scores[np.arange(len(q)),targets]
    ranks=1+(scores>ts[:,None]).sum(1)+((scores==ts[:,None])&(np.arange(len(g))[None,:]<targets[:,None])).sum(1)
    return ranks

def macro_metrics(per_category):
    require(set(per_category)==set(CATEGORIES),'Macro requires three categories')
    overall={f'R@{k}':float(np.mean([per_category[c][f'R@{k}'] for c in CATEGORIES])) for k in (1,5,10,50)}
    return {'overall':overall,'Mean':(overall['R@10']+overall['R@50'])/2}

def pure_tests():
    cases=[(['red','long'],None,None,'Red and long'),(['','long'],None,'single_empty_caption',' and long'),
           (['red',''],None,'single_empty_caption','Red and '),([' .?, ','long'],None,'single_empty_caption',' and long'),
           ([' \t ','.?, '],'both_empty_captions',None,None),(['',''],'both_empty_captions',None,None),
           (['red',1],'schema_error',None,None),(['red'],'schema_error',None,None),('red','schema_error',None,None)]
    for caps,error,warning,text in cases:
        require(assess_captions(caps)==(error,warning),'Caption assessment mismatch')
        if text is not None:require(caption_text(*caps)==text,'Caption policy mismatch')
    require(caption_text(' .Red?, ',' BLUE. ')=='Red and BLUE','Caption normalization strips only .?, and ASCII space at both ends')
    require(normalize_caption('\tblue\t')=='\tblue\t','Tabs must not be silently stripped by policy')
    q=np.array([[1,0],[0,1]],np.float32);g=np.array([[1,0],[1,0],[0,1]],np.float32)
    require(exact_ranks(q,g,[1,2]).tolist()==[2,1],'Manual rank/tie test failed')
    rng=np.random.default_rng(42);a=rng.normal(size=(7,8)).astype(np.float32);b=rng.normal(size=(13,8)).astype(np.float32);targets=np.arange(7)
    a/=np.linalg.norm(a,axis=-1,keepdims=True);b/=np.linalg.norm(b,axis=-1,keepdims=True)
    orders=np.argsort(-(a@b.T),axis=1,kind='stable')
    expected=[int(np.flatnonzero(row==t)[0])+1 for row,t in zip(orders,targets)]
    require(exact_ranks(a,b,targets).tolist()==expected,'Exact rank and stable sorting disagree')
    cat={c:{'R@1':0.,'R@5':10.,'R@10':20.+i*10,'R@50':60.+i*10} for i,c in enumerate(CATEGORIES)}
    require(macro_metrics(cat)['overall']['R@10']==30 and macro_metrics(cat)['Mean']==50,'Macro/Mean test failed')
    ids=['a','a','b','c'];mask=positive_mask(ids,torch.device('cpu'));require(mask[0,1] and not mask[0,2],'Duplicate target positive mask failed')
    x=unit(torch.arange(1,17,dtype=torch.float32).reshape(4,4));y=x.flip(0)
    logits=x@y.T/0.07
    manual=[]
    for scores,p in ((logits,mask),(logits.T,mask.T)):
        manual.append(torch.stack([-F.log_softmax(scores[i],dim=0)[p[i]].mean() for i in range(4)]).mean())
    require(torch.allclose(retrieval_loss(x,y,ids,.07),sum(manual)/2),'Multi-positive reduction mismatch')
    student=torch.randn(4,8,requires_grad=True);teacher=torch.randn(4,8,requires_grad=True)
    value=cosine_alignment(student,teacher);value.backward()
    require(teacher.grad is None and student.grad is not None and torch.isfinite(student.grad).all(),'Cosine teacher detach')
    require(torch.allclose(value.detach(),1-F.cosine_similarity(student.detach(),teacher.detach()).mean()),'Cosine reduction')
    return {'caption_cases':len(cases)+2,'exact_rank_tie_and_stable_sort':True,'macro_mean':True,'duplicate_target_multi_positive':True,'cosine_alignment_detach':True}


In [ ]:
# training.py — code nhúng, không phụ thuộc file local
def checkpoint_save(path,state,signature):
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    require(state['signature']==signature,'Checkpoint signature differs')
    with tempfile.NamedTemporaryFile(dir=path.parent,suffix='.pt',delete=False) as f:temp=Path(f.name)
    try:
        torch.save(state,temp);meta={'signature':signature,'sha256':file_hash(temp),'phase':state.get('phase'),'step':state.get('step')}
        write_json(path.with_suffix('.pending.json'),meta);os.replace(temp,path);write_json(path.with_suffix('.json'),meta)
    finally:
        if temp.exists():temp.unlink()

def checkpoint_load(path,signature):
    path=Path(path);marker=path.with_suffix('.json');pending=path.with_suffix('.pending.json');sha=file_hash(path)
    if pending.is_file():
        p=read_json(pending)
        if p.get('sha256')==sha:
            require(p['signature']==signature,'Pending checkpoint signature');write_json(marker,p)
    m=read_json(marker);require(m['signature']==signature and m['sha256']==sha,'Checkpoint checksum/config mismatch')
    state=torch.load(path,map_location='cpu',weights_only=False)
    require(state['signature']==signature,'Checkpoint payload identity differs');return state

def sync_checkpoint(local,destination,signature):
    destination=Path(destination);destination.parent.mkdir(parents=True,exist_ok=True)
    marker=read_json(Path(local).with_suffix('.json'));require(marker['signature']==signature,'Checkpoint sync signature')
    write_json(destination.with_suffix('.pending.json'),marker)
    temp=destination.with_suffix('.copying');shutil.copyfile(local,temp);require(file_hash(temp)==marker['sha256'],'Drive checkpoint sync corrupt')
    os.replace(temp,destination);write_json(destination.with_suffix('.json'),marker)

def epoch_batches(frame,batch,seed,epoch):
    ids=np.asarray(sorted(frame.query_id),dtype=object);order=np.random.default_rng(seed+epoch).permutation(ids)
    require(len(ids)>=2 and batch>=2,'Contrastive batch too small')
    batches=[order[i:i+batch].tolist() for i in range(0,len(ids),batch)]
    if len(batches)>1 and len(batches[-1])==1:batches[-2].extend(batches.pop())
    return batches

def train_adapter(variant,source,store,frozen,base=None,hashes=None):
    require(variant in ('retrieval_only','retrieval_alignment'),'Unknown adapter trial')
    root=RUN/variant;root.mkdir(exist_ok=True)
    tests=read_json(RUN/'test_report.json');require(not tests['failed'] and tests['locked_batch']==CFG['batch_size'],'Adapter preflight not passed')
    require(read_json(RUN/'baseline'/'verification.json')['passed'],'Baseline reproduction not passed')
    require(read_json(RUN/'locked_training_config.json')=={'config':CFG,'code_sha256':CODE_SHA256,'cache_fingerprint':store.key,
        'projection_hash':hashlib.sha256(frozen.projection.weight.detach().cpu().numpy().tobytes()).hexdigest()},'Training config changed after profiling')
    weight=CFG['alignment_weight'] if variant=='retrieval_alignment' else 0.
    model=adapter_initial(frozen.projection.weight);initial_sha=tensor_hash(model);params=sum(p.numel() for p in model.parameters() if p.requires_grad)
    require(params<5_000_000,'Adapter exceeds budget')
    binding={'variant':variant,'alignment_weight':weight,'config':CFG,'code':CODE_SHA256,'cache':store.key,'base_checkpoint':EXPECTED_BASE_MARKER['sha256'],
        'initial_sha256':initial_sha,'subset_sha256':source['subset_sha256'],'validation_sha256':source['validation_sha256'],'gallery_sha256':source['gallery_sha256']}
    signature=fingerprint(binding);write_json(root/'config.json',{'signature':signature,'binding':binding,'trainable_parameters':params,
        'gate_initialization':0.,'objective':'symmetric multi-positive InfoNCE + fixed lambda * cosine alignment; not TME KL',
        'scheduler':None,'scaler':None,'sync_policy':'local+Drive every16 steps and each epoch; at most15 steps lost on disappearing runtime'},immutable=True)
    opt=torch.optim.AdamW(model.parameters(),lr=CFG['learning_rate'],weight_decay=CFG['weight_decay'],betas=(.9,.999),eps=1e-8)
    local=LOCAL/'checkpoints'/variant;local.mkdir(parents=True,exist_ok=True)
    state={'signature':signature,'phase':variant,'step':0,'epoch':0,'cursor':0,'finished':False,'history':[],'metrics':[],
        'best_mean':None,'best_epoch':None,'train_compute_seconds':0.,'validation_seconds':0.,'config':CFG,'binding':binding,'scheduler':None,'scaler':None}
    candidates=[]
    for p in (local/'last.pt',root/'last.pt'):
        if p.exists():
            s=checkpoint_load(p,signature);require(s['binding']==binding and s['config']==CFG,'Resume binding/config differs');candidates.append(s)
    if candidates:
        state=max(candidates,key=lambda x:x['step']);model.load_state_dict(state['model'],strict=True);opt.load_state_dict(state['optimizer']);restore_rng(state['rng'])
        require(len(state['history'])==state['step'],'Resume history/cursor disagree')
        print('Resume',variant,'epoch',state['epoch'],'cursor',state['cursor'],'step',state['step'])
    else:seed_all(42)
    del candidates
    def commit(best=False):
        state.update(model=model.state_dict(),optimizer=opt.state_dict(),rng=rng_state())
        checkpoint_save(local/'last.pt',state,signature);sync_checkpoint(local/'last.pt',root/'last.pt',signature)
        if best:sync_checkpoint(local/'last.pt',root/'best.pt',signature)
        write_csv(root/'training_history.csv',pd.DataFrame(state['history']));write_json(root/'metric_history.json',state['metrics'])
        write_json(root/'status.json',{'finished':state['finished'],'step':state['step'],'epoch':state['epoch'],'best_epoch':state['best_epoch'],'best_mean':state['best_mean'],
            'train_compute_seconds':state['train_compute_seconds'],'validation_seconds':state['validation_seconds'],'trainable_parameters':params,'initial_sha256':initial_sha})
    indexed=source['subset'].set_index('query_id',drop=False)
    try:
        for epoch in range(state['epoch'],CFG['max_epochs']):
            if state['finished']:break
            order=epoch_batches(source['subset'],CFG['batch_size'],42,epoch)
            require(state['cursor']<=len(order),'Resume data cursor outside epoch')
            if DEVICE.type=='cuda':torch.cuda.reset_peak_memory_stats()
            for cursor in range(state['cursor'],len(order)):
                rows=indexed.loc[order[cursor]];require(rows.query_id.tolist()==order[cursor],'Batch mapping/order')
                b=make_batch(rows,store);model.train();opt.zero_grad(set_to_none=True)
                if DEVICE.type=='cuda':torch.cuda.synchronize()
                start=time.perf_counter();total,r,a=objective(model,b,weight);total.backward();norms=gradient_summary(model)
                g=torch.nn.utils.clip_grad_norm_(model.parameters(),CFG['gradient_clip'],error_if_nonfinite=True);opt.step()
                if DEVICE.type=='cuda':torch.cuda.synchronize()
                elapsed=time.perf_counter()-start;state['train_compute_seconds']+=elapsed
                values=torch.stack([r.detach(),a.detach(),total.detach(),g.detach(),torch.tanh(model.gate_ri).detach(),torch.tanh(model.gate_ta).detach()]).cpu().tolist()
                state['step']+=1;state['cursor']=cursor+1
                state['history'].append({'step':state['step'],'epoch':epoch+1,'cursor':cursor+1,'batch_ids_sha256':fingerprint(order[cursor]),
                    'loss_retrieval':values[0],'loss_cosine':values[1],'loss_total':values[2],'gradient_norm':values[3],
                    'gate_RI':values[4],'gate_TA':values[5],'lr':opt.param_groups[0]['lr'],'compute_seconds':elapsed,**norms})
                if state['step']%CFG['checkpoint_every_steps']==0:commit()
            model.eval();probe=alignment_influence(model,make_batch(stratified_sample(source['subset'],CFG['batch_size'],42),store),weight)
            start=time.perf_counter();metric,pred=evaluate_adapter(model,source,store);eval_seconds=time.perf_counter()-start;state['validation_seconds']+=eval_seconds
            require(metric['n_query']==len(source['validation']),'Incomplete validation')
            improved=state['best_mean'] is None or metric['Mean']>state['best_mean']
            if improved:state.update(best_mean=metric['Mean'],best_epoch=epoch+1)
            state['metrics'].append({'epoch':epoch+1,'metrics':metric,'alignment_gradient_probe':probe,'validation_seconds':eval_seconds,
                'peak_allocated_bytes':torch.cuda.max_memory_allocated() if DEVICE.type=='cuda' else None})
            write_csv(root/f'epoch_{epoch+1:02d}_per_query.csv',pred)
            state.update(epoch=epoch+1,cursor=0,finished=epoch+1>=CFG['max_epochs']);commit(best=improved)
            print(variant,'epoch',epoch+1,'Dev Mean',metric['Mean'],'best epoch',state['best_epoch'])
        if base is not None:assert_backbones_frozen(base,frozen,hashes)
        return {k:v for k,v in state.items() if k not in ('model','optimizer','rng')}
    except BaseException:
        write_bytes(root/'traceback.txt',traceback.format_exc().encode());raise
    finally:
        opt.zero_grad(set_to_none=True)

def final_summary():
    baseline=read_json(RUN/'baseline'/'verification.json');rows=[]
    m=baseline['metrics'];rows.append({'configuration':'frozen_conventional','finished':baseline['passed'],'best_epoch':3,'R@10':m['overall']['R@10'],'R@50':m['overall']['R@50'],'Mean':m['Mean'],
        'delta_Mean':0.,'train_compute_seconds':0.,'validation_seconds':baseline['elapsed_seconds'],'trainable_parameters':0})
    for variant in ('retrieval_only','retrieval_alignment'):
        row={'configuration':variant,'finished':False,'best_epoch':None,'R@10':None,'R@50':None,'Mean':None,'delta_Mean':None,'train_compute_seconds':None,'validation_seconds':None,'trainable_parameters':None}
        if (RUN/variant/'status.json').is_file():
            s=read_json(RUN/variant/'status.json');history=read_json(RUN/variant/'metric_history.json')
            row.update({k:s[k] for k in ('finished','best_epoch','train_compute_seconds','validation_seconds','trainable_parameters')})
            if history:
                best=max(history,key=lambda h:(h['metrics']['Mean'],-h['epoch']));b=best['metrics']
                row.update({'R@10':b['overall']['R@10'],'R@50':b['overall']['R@50'],'Mean':b['Mean'],'delta_Mean':b['Mean']-m['Mean']})
        rows.append(row)
    write_csv(RUN/'comparison.csv',pd.DataFrame(rows))
    write_json(RUN/'trial_summary.json',{'results':rows,'cache_cost':read_json(RUN/'cache_timing.json'),'seed':42,'subset':2048,'official_val_used':False,
        'final_benchmark_started':False,'heads_trained':False,'statistical_significance_claimed':False,'speedup_vs_full_TME_claimed':False,
        'note':'One-seed screening only; report negative deltas. Cache/encode cost is separate from adapter train compute and validation; checkpoint I/O is not included in compute_seconds.'})
    print(pd.DataFrame(rows).round(5).to_string(index=False));return rows

@torch.inference_mode()
def export_enhanced(model,frame,store,destination):
    require(frame.query_id.is_unique and set(frame.official_split)=={'train'},'Export development only')
    require(set(frame.query_id)<=set(SOURCE['subset'].query_id)|set(SOURCE['validation'].query_id),'Export outside trial manifests')
    binding={'model_sha256':tensor_hash(model),'cache':store.key,'query_ids':frame.query_id.tolist(),'base_checkpoint':EXPECTED_BASE_MARKER['sha256'],'sequence':'post-finalLN conventional + gated RI/TA residual; not composed vector'}
    key=fingerprint(binding);root=Path(destination)/key;root.mkdir(parents=True,exist_ok=True);write_json(root/'index.json',binding,immutable=True);model.eval()
    for start in range(0,len(frame),128):
        f=frame.iloc[start:start+128];p=root/f'{start:07d}.npz';marker=p.with_suffix('.json')
        if p.exists():
            require(marker.exists() and read_json(marker)['fingerprint']==key and read_json(marker)['sha256']==file_hash(p),'Existing export corrupt');continue
        arrays={k:[] for k in ('sequence','mask','eos')}
        for offset in range(0,len(f),CFG['eval_batch_size']):
            b=make_batch(f.iloc[offset:offset+CFG['eval_batch_size']],store,training=False)
            h,m,e=model.enhanced_text_sequence(b['reference_features'],b['text_features'],b['mask'],b['eos'])
            for name,v in [('sequence',h),('mask',m),('eos',e)]:arrays[name].append(v.cpu().numpy())
        temp=p.with_suffix('.partial.npz');np.savez(temp,**{k:np.concatenate(v) for k,v in arrays.items()},query_id=f.query_id.to_numpy(dtype=str))
        os.replace(temp,p);write_json(marker,{'sha256':file_hash(p),'fingerprint':key,'query_ids':f.query_id.tolist()})
    return root

def load_best_adapter(variant):
    root=RUN/variant;config=read_json(root/'config.json');b=config['binding']
    require(config['signature']==fingerprint(b) and b['variant']==variant,'Export checkpoint binding fingerprint differs')
    require(b['config']==CFG and b['code']==CODE_SHA256 and b['cache']==CACHE.key and b['base_checkpoint']==EXPECTED_BASE_MARKER['sha256'],'Export checkpoint provenance differs')
    state=checkpoint_load(root/'best.pt',config['signature']);model=adapter_initial(FROZEN.projection.weight)
    model.load_state_dict(state['model'],strict=True);model.eval();return model


In [ ]:
# tests_runtime.py — code nhúng, không phụ thuộc file local
def query_from_batch(model,batch,**kw):
    return model(batch['reference_features'],batch['text_features'],batch['mask'],batch['eos'],**kw)

def batch_slice(batch,n):
    result={k:({p:v[:n] for p,v in value.items()} if isinstance(value,dict) else value[:n]) for k,value in batch.items()}
    return result

def adapter_smoke(projection,batch,weight):
    saved=rng_state();model=adapter_initial(projection);before=tensor_hash(model);model.eval()
    try:
        with torch.no_grad():
            index=torch.arange(len(batch['eos']),device=DEVICE)
            baseline=unit(unit(F.linear(batch['text_features'][index,batch['eos']],projection))+unit(batch['reference_features']['pooled']))
            q=query_from_batch(model,batch);require(torch.allclose(q,baseline,atol=CFG['gate_equivalence_atol'],rtol=0),'Zero gates changed baseline')
            for n in (1,len(batch['eos'])):
                b=batch_slice(batch,n);value=query_from_batch(model,b);require(value.shape==(n,projection.shape[0]),'B1/normal output shape');finite(value,'query')
            changed={**batch,'target':batch['target'].flip(0)}
            require(torch.equal(query_from_batch(model,batch),query_from_batch(model,changed)),'Target altered query')
            dirty={**batch,'text_features':batch['text_features'].clone()};dirty['text_features'][batch['mask']==0]=12345.
            require(torch.equal(query_from_batch(model,batch),query_from_batch(model,dirty)),'Padding altered EOS pooling')
        opt=torch.optim.AdamW(model.parameters(),lr=CFG['learning_rate'],weight_decay=CFG['weight_decay'])
        stages=[]
        for step in range(4):
            opt.zero_grad(set_to_none=True);teacher=batch['target'].detach().clone().requires_grad_()
            b={**batch,'target':teacher};total,r,a=objective(model,b,weight);total.backward()
            require(teacher.grad is None,'Teacher received gradient')
            require(all(p.grad is not None and bool(torch.isfinite(p.grad).all()) for p in model.parameters()),'Missing/nonfinite adapter grad')
            norms=gradient_summary(model);stages.append(norms)
            if step==0:
                require(norms['RI']==0,'Zero-gated RI unexpectedly active before first update')
                if weight==0:require(norms['TA']==0,'Zero-gated retrieval TA unexpectedly active')
                else:require(norms['TA']>0,'Cosine TA should receive auxiliary gradient at zero gate')
            torch.nn.utils.clip_grad_norm_(model.parameters(),CFG['gradient_clip'],error_if_nonfinite=True);opt.step()
        require(stages[-1]['RI']>0 and stages[-1]['TA']>0,'Residual branches remain disconnected after gate updates')
        with torch.no_grad():require(torch.allclose(query_from_batch(model,batch,gates_off=True),baseline,atol=CFG['gate_equivalence_atol'],rtol=0),'Forced-off trained gates changed baseline')
        influence=alignment_influence(model,batch,weight)
        # Exact optimizer + RNG save/load continuation on this temporary model.
        with tempfile.TemporaryDirectory() as d:
            path=Path(d)/'smoke.pt';sig=fingerprint({'test':'adapter-smoke','weight':weight})
            checkpoint_save(path,{'signature':sig,'phase':'test','step':4,'model':model.state_dict(),'optimizer':opt.state_dict(),'rng':rng_state()},sig)
            def step_once():
                opt.zero_grad(set_to_none=True);loss=objective(model,batch,weight)[0]*(1+.01*(torch.rand((),device=DEVICE)+random.random()+float(np.random.random())));loss.backward();opt.step()
            step_once();expected={k:v.detach().clone() for k,v in model.state_dict().items()};expected_rng=rng_state()
            state=checkpoint_load(path,sig);model.load_state_dict(state['model'],strict=True);opt.load_state_dict(state['optimizer']);restore_rng(state['rng']);step_once()
            require(all(torch.equal(v,model.state_dict()[k]) for k,v in expected.items()),'Optimizer continuation differs after resume')
            actual_rng=rng_state()
            require(torch.equal(expected_rng['torch'],actual_rng['torch']) and expected_rng['python']==actual_rng['python'] and np.array_equal(expected_rng['numpy'][1],actual_rng['numpy'][1]) and expected_rng['numpy'][2:]==actual_rng['numpy'][2:],'Resume RNG differs')
            if expected_rng['cuda'] is not None:require(all(torch.equal(a,b) for a,b in zip(expected_rng['cuda'],actual_rng['cuda'])),'CUDA RNG differs')
        return {'passed':True,'weight':weight,'gate_off_equivalence':True,'batch_sizes':[1,len(batch['eos'])],'target_independence':True,'teacher_detached':True,
            'padding_pooling_invariant':True,'gradient_stages':stages,'alignment_influence':influence,'optimizer_rng_resume':True,
            'trainable_parameters':sum(p.numel() for p in model.parameters() if p.requires_grad),'temporary_initial_sha256':before}
    finally:restore_rng(saved)

def profile_and_preflight(source,store,frozen,base,hashes):
    require(read_json(RUN/'baseline'/'verification.json')['passed'],'Baseline must reproduce before adapter tests/training')
    saved=rng_state();report={'passed':[],'failed':[],'skipped':[],'profiling':[]}
    try:
        report['passed'].append(pure_tests())
        report['passed'].append(storage_tests())
        for weight in (0.,CFG['alignment_weight']):
            batch=make_batch(stratified_sample(source['subset'],CFG['batch_size'],42),store)
            report['passed'].append(adapter_smoke(frozen.projection.weight,batch,weight));del batch
        for n in CFG['profile_batches']:
            model=None;batch=None;opt=None
            try:
                model=adapter_initial(frozen.projection.weight);batch=make_batch(stratified_sample(source['subset'],n,42),store)
                opt=torch.optim.AdamW(model.parameters(),lr=CFG['learning_rate'],weight_decay=CFG['weight_decay'])
                if DEVICE.type=='cuda':torch.cuda.reset_peak_memory_stats()
                for _ in range(2):
                    opt.zero_grad(set_to_none=True);objective(model,batch,CFG['alignment_weight'])[0].backward();torch.nn.utils.clip_grad_norm_(model.parameters(),CFG['gradient_clip'],error_if_nonfinite=True);opt.step()
                if DEVICE.type=='cuda':torch.cuda.synchronize()
                start=time.perf_counter()
                for _ in range(3):
                    opt.zero_grad(set_to_none=True);objective(model,batch,CFG['alignment_weight'])[0].backward();torch.nn.utils.clip_grad_norm_(model.parameters(),CFG['gradient_clip'],error_if_nonfinite=True);opt.step()
                if DEVICE.type=='cuda':torch.cuda.synchronize()
                elapsed=time.perf_counter()-start
                report['profiling'].append({'batch':n,'passed':True,'examples_per_second':3*n/elapsed,'three_steps_seconds':elapsed,'scope':'adapter compute/backward/clip/AdamW only; excludes batch I/O and checkpoint sync',
                  'peak_allocated_bytes':torch.cuda.max_memory_allocated() if DEVICE.type=='cuda' else None})
            except torch.cuda.OutOfMemoryError:
                report['profiling'].append({'batch':n,'passed':False,'reason':'CUDA OOM'})
                if n==CFG['batch_size']:raise RuntimeError('Configured batch OOM; select new experiment ID/batch after reviewing profile. No automatic batch/precision change.') from None
            finally:
                del model,batch,opt
                if DEVICE.type=='cuda':torch.cuda.empty_cache()
        require(any(p['batch']==CFG['batch_size'] and p['passed'] for p in report['profiling']),'Configured batch was not profiled')
        counts=[r['trainable_parameters'] for r in report['passed'] if 'trainable_parameters' in r]
        require(counts and max(counts)<5_000_000,'Adapter exceeds parameter budget')
        assert_backbones_frozen(base,frozen,hashes)
        report['passed'].append({'backbone_hashes_unchanged':True,'frozen_eval_no_grad':True})
        report['locked_batch']=CFG['batch_size'];report['batch_policy']='Record 32/64 throughput; keep configured shared batch. Changing requires new experiment ID.'
        write_json(RUN/'locked_training_config.json',{'config':CFG,'code_sha256':CODE_SHA256,'cache_fingerprint':store.key,'projection_hash':hashlib.sha256(frozen.projection.weight.detach().cpu().numpy().tobytes()).hexdigest()},immutable=True)
    except BaseException:
        report['failed'].append(traceback.format_exc());raise
    finally:
        restore_rng(saved);write_json(RUN/'test_report.json',report)
    print('Adapter preflight đạt; trainable params:',counts[0],'| batch đã khóa:',CFG['batch_size'])
    return report

def storage_tests():
    saved=rng_state()
    try:
        with tempfile.TemporaryDirectory() as d:
            root=Path(d);spec={'x':{'ids':['a','b'],'shape':[3],'dtype':'float32'}}
            st=TrialStore(root/'remote',root/'local',{'fixture':42},spec,2);st.prepare('x',lambda ids:np.ones((len(ids),3),np.float32))
            require(st.take('x',['b','a']).shape==(2,3),'Cache ID read')
            marker=next((st.root/'x').glob('*.json'));original=marker.read_bytes();m=read_json(marker);m['fingerprint']='wrong';marker.write_text(json.dumps(m))
            try:st.prepare('x',lambda ids:None)
            except RuntimeError:pass
            else:raise AssertionError('Wrong fingerprint accepted')
            marker.write_bytes(original);data=next((st.root/'x').glob('*.npy'));raw=data.read_bytes();data.write_bytes(raw[:-2]+b'xx')
            try:st.prepare('x',lambda ids:None)
            except RuntimeError:pass
            else:raise AssertionError('Corrupt cache accepted')
            data.write_bytes(raw);pending=marker.with_suffix('.pending.json');pending.write_bytes(original);marker.unlink();st.prepare('x',lambda ids:None)
            require(marker.is_file(),'Pending commit not recovered')
        return {'cache_corruption_blocked':True,'wrong_fingerprint_blocked':True,'cache_pending_commit_recovery':True}
    finally:restore_rng(saved)


## 2. Strict checkpoint và preflight encoder thật
Chỉ đọc checkpoint/hash/config đã kiểm kê, không dò checkpoint để làm khớp điểm. Nạp pretrained đúng revision để lấy vision/tokenizer, strict-load state conventional, xác minh schema/epoch/selection, đóng băng và forward thật trước cache. Hash backbone tại ranh giới giai đoạn, không theo batch.

In [ ]:
init_trial()
with stage('source_and_checkpoint_preflight'):
    SOURCE=inspect_sources()
    BASE,FROZEN,TOKENIZER,PROCESSOR,BACKBONE_HASHES=load_frozen_checkpoint(SOURCE)
    assert_backbones_frozen(BASE,FROZEN,BACKBONE_HASHES)


## 3. Cache đầu vào nhất quán và tái đánh giá baseline
Cache text `[N,77,512]` sau finalLN của **trained conventional** + mask/input-ID/EOS; reference final tokens `[R,50,768]` và pooled `L2(projection(postLN(CLS)))`; target/gallery pooled `[G,512]` trong cùng không gian vision frozen. Tái dùng image shards preparation theo index/ID/SHA/dtype/contract; không tải ảnh/dataset nếu feature package hợp lệ. Không đọc old text arrays hoặc multi-layer cache.

Shard có fingerprint, commit/pending marker, checksum, shape/finite. Kiểm tra **mọi query mask/EOS/padding** trước train, local mmap trong batch. Cache encode và thời gian baseline lưu riêng. Baseline dùng exact cosine, giữ reference trong gallery, tie theo gallery_index tăng; macro3category, Mean=(R10+R50)/2. Dừng nếu metric lệch quá **0,01 điểm phần trăm** đã cố định; lưu rank comparison để chẩn đoán, không tăng tolerance/chọn checkpoint khác.


In [ ]:
with stage('input_cache_and_baseline'):
    started=time.perf_counter()
    CACHE=prepare_cache(SOURCE,BASE,FROZEN,TOKENIZER)
    write_json(RUN/'cache_timing.json',{'elapsed_seconds':time.perf_counter()-started,'scope':'source metadata/hash/copy + trained text encoding + final image feature reuse + exhaustive cache audit; not adapter training','fingerprint':CACHE.key})
    BASELINE_REPORT=verify_baseline(SOURCE,CACHE,FROZEN)
    assert_backbones_frozen(BASE,FROZEN,BACKBONE_HASHES)
print('Baseline tái hiện đạt:',BASELINE_REPORT['metrics']['Mean'],'| deviation pp:',BASELINE_REPORT['deviation_pp']['Mean'])


## 4. Adapter, kiểm thử và profiling 32/64
Text conventional **không đổi**: `H_enh=(H+tanh(g_RI)ΔRI+tanh(g_TA)ΔTA)*mask`. Chỉ nhánh residual có LN/MLP. RI: text512→128 cross-attend final reference768→128,4head, MLP→512. TA: text LN→MLP512→128→512. Hai gate scalar khởi tạo0. Pool EOS thật → **trained text projection frozen** → L2; query=L2(text+L2(reference pooled)), chỉ cộng reference một lần. API query chỉ reference/text/mask/EOS, không target. Đếm tham số thực tế và chặn nếu ≥5triệu.

Retrieval symmetric multi-positive InfoNCE T=0,07 theo target ID. Alignment **cosine**, `mean(1−cos(P(H+ΔTA)_EOS,detach(target)))`, khác KL bản TME trước. Trọng số B=0, C=0,1 cố định. B/C cùng kiến trúc, initialization, thứ tự batch và ngân sách. TA/RI retrieval gradient có thể bằng0 ở step0 do gate; test kiểm tra cơ chế ở step0 và sau4step, không bắt nonzero mọi tensor ngay đầu.

Smoke/profiling dùng model/RNG tạm; kiểm tra gate-off, target detach/independence, padding, B1/batch thường, duplicate target, ranking/macro, corruption/fingerprint, save-load và optimizer/RNG continuation. Profile32/64 x3step sau2warmup. **Giữ batch32 mặc định nếu đạt**, ghi throughput64; muốn chọn64 cần CFG/experiment mới trước train, không tự đổi giữa lượt. FP32, không AMP/TF32. Nếu OOM batch đã chọn, dừng để xem report.


In [ ]:
with stage('adapter_preflight_and_profile'):
    TEST_REPORT=profile_and_preflight(SOURCE,CACHE,FROZEN,BASE,BACKBONE_HASHES)


## 5. Hai trial adapter
AdamW LR1e-4, decay0,01, clip1, batch chung đã khóa, tối đa3epoch, eval mỗi epoch. Best Dev Mean cao nhất, tie giữ epoch sớm nhất. Scheduler/scaler=None, constant LR. Baseline A không training; B/C khởi tạo mới cùng seed, không warm-start nhau. Lưu loss/gradient/gates, ảnh hưởng gradient alignment, best/last, per-category/per-query.

Checkpoint local atomic rồi sync Drive mỗi16step và cuối epoch; nếu runtime biến mất có thể mất tối đa15step chưa commit. Resume đúng config/code/cache/base hash, optimizer/RNG/epoch/cursor; không đổi batch/precision, không đọc output cũ như metric mới.


In [ ]:
RESULTS={}
for variant in ('retrieval_only','retrieval_alignment'):
    with stage('train_'+variant):
        RESULTS[variant]=train_adapter(variant,SOURCE,CACHE,FROZEN,BASE,BACKBONE_HASHES)


## 6. Bàn giao kết quả và điểm dừng
Bảng baseline/B/C gồm R10/R50/Mean/delta, parameter count, train compute và validation time; chi phí cache lưu riêng. Metric chưa chạy=null. Báo cả delta âm. Một seed/subset không chứng minh ý nghĩa thống kê; không so tốc độ với TME full khác phạm vi. Không tự mở rộng seed/full train, không encode official-val.

API chuẩn bị cho head sau: `load_best_adapter(variant)` strict-load best adapter; `model.enhanced_text_sequence(reference_features,text_features,mask,eos)` trả `[B,77,512]`, mask và EOS, **không composed vector**. `export_enhanced(model, frame, CACHE, destination)` là opt-in, kiểm tra ID development và provenance; chưa gọi/tích hợp XAA/DVR ở đây.

Runtime mới: chạy từ cell đầu cùng CFG/experiment; các cache/checkpoint đúng fingerprint được tái dùng. Nếu bootstrap yêu cầu restart, lưu notebook/traceback trước. **Notebook mới chưa chạy trên Colab/GPU**; local tests dùng tiny random CLIP và fixture, không thay thế preflight thật. Không có output cũ/metric thử nghiệm giả trong notebook.


In [ ]:
with stage('final_summary'):
    assert_backbones_frozen(BASE,FROZEN,BACKBONE_HASHES)
    COMPARISON=final_summary()
